# GRAST — WP07.6.2 Curriculum SFT trên Kaggle

Recipe và model giống bản Colab: **400 copy →1.200 copy+single →1.200 copy+single+multi updates**, chọn model theo48 câu validation tự sinh. Chưa có kết quả train đầy đủ chứng minh reasoning cải thiện.

1. Bật **Internet**, chọn **GPU** trong Settings. Notebook giữ PyTorch sẵn có, dùng `cuda:0` (kể cả có2 GPU), không DDP.
2. Add Input một Kaggle Dataset chứa bundle `grast-wp07-curriculum-0a891fda1dc3.zip` và **SFT500** `final.pt` từ local. Summary ZIP không chứa model weights. ZIP hoặc layout Kaggle tự giải nén đều được kiểm hash.
3. Giữ `SMOKE_ONLY=True`, `RESUME=""`; chọn **Save Version → Save & Run All**. Đây là phiên sạch, không dựa vào file của editor và không cần upload/mount/download tương tác.
4. Sau smoke, đặt `SMOKE_ONLY=False` để train đủ. Có thể chạy lại từ SFT500 hoặc gắn Notebook Output smoke làm Input và điền `RESUME` đúng đường dẫn latest.pt để tiếp tục.

Input weights phải là `D:/GRAST/artifacts/checkpoints/wp07-cot-sft-math-500/final.pt`, không phải pretraining hoặc diagnostic checkpoint.


In [ ]:
import hashlib
import importlib.util
import json
import os
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONUNBUFFERED="1")
SOURCE_COMMIT = "2383ab9a2900dc276eec7305a7e7d8cdfc5f2903"
TOTAL_UPDATES = 2800
SMOKE_ONLY = True
RESUME = ""  # Exact full curriculum latest.pt; NOT selected_model.pt or old SFT final.pt.
SFT_CHECKPOINT = ""  # Optional explicit path; empty discovers SFT500 by SHA256.
BUNDLE_PATH = ""  # Optional explicit ZIP path or expanded bundle directory.

BUNDLE_NAME = "grast-wp07-curriculum-0a891fda1dc3.zip"
BUNDLE_SHA256 = "02c9e6a9d969365921d00cc119e614295a25f1d1bde53fb7808141f2ffcee217"
MANIFEST_SHA256 = "6a01d2b3a89139ed740c69e533e62bd5bddb9e8a8ec484cbc69cc89f07783d25"
SFT_SHA256 = "5f75909261c79e9f7f0c6e37a7e9d76ba5670be8b4be5133b6e148e0cd06ff42"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/grast-wp07-curriculum")
OUTPUT_ROOT = WORK_ROOT / "runs"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CONSOLE_LOG = WORK_ROOT / "console.log"
LAST_RUN_DIRECTORY = None
# Source and extracted data stay outside working to keep Notebook Output small.
SCRATCH_ROOT = Path(tempfile.mkdtemp(prefix="grast-curriculum-kaggle-"))
REPO = SCRATCH_ROOT / "source"
CHILD_ENV = os.environ.copy()
CHILD_ENV["PYTHONPATH"] = str(REPO / "src") + os.pathsep + CHILD_ENV.get("PYTHONPATH", "")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run_logged(command, *, cwd=None):
    global LAST_RUN_DIRECTORY
    line = "RUN: " + " ".join(map(str, command))
    print(line, flush=True)
    with CONSOLE_LOG.open("a", encoding="utf-8") as log:
        log.write(line + "\n")
        log.flush()
        with subprocess.Popen(
            list(map(str, command)),
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            bufsize=1,
            env=CHILD_ENV,
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                    try:
                        record = json.loads(line)
                    except (ValueError, TypeError):
                        continue
                    if isinstance(record, dict) and record.get("event") in (
                        "curriculum_started",
                        "curriculum_finished",
                    ):
                        LAST_RUN_DIRECTORY = record["run_directory"]
                status = process.wait()
            except BaseException:
                if process.poll() is None:
                    process.send_signal(signal.SIGINT)
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                raise
        if status:
            print("Last run directory:", LAST_RUN_DIRECTORY, flush=True)
            raise subprocess.CalledProcessError(status, command)
    return LAST_RUN_DIRECTORY

In [ ]:
run_logged(
    [
        "git",
        "clone",
        "--branch",
        "feat/wp07-6-2-kaggle",
        "--single-branch",
        "https://github.com/CuongBien/GRAST.git",
        REPO,
    ]
)
run_logged(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "--no-deps", REPO])
if importlib.util.find_spec("matplotlib") is None:
    run_logged([sys.executable, "-u", "-m", "pip", "install", "matplotlib>=3.9,<4"])
# Project operations use fresh subprocesses and explicit PYTHONPATH, so an old
# llm_scratch import in this kernel cannot select another checkout.
run_logged(
    [
        sys.executable,
        "-u",
        "-c",
        "import llm_scratch; print('Project source:', llm_scratch.__file__, flush=True)",
    ]
)

In [ ]:
def find_bundle(input_root, explicit_path=""):
    root = Path(input_root)
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError("Explicit bundle path does not exist: " + str(path))
        candidates = [("expanded" if path.is_dir() else "zip", path)]
    else:
        candidates = [("zip", p) for p in sorted(root.rglob(BUNDLE_NAME)) if p.is_file()]
        for manifest in sorted(root.rglob("bundle_manifest.json")):
            if manifest.stat().st_size > 65536:
                continue
            try:
                meta = json.loads(manifest.read_text(encoding="utf-8"))
            except (ValueError, UnicodeError):
                continue
            if isinstance(meta, dict) and meta.get("scope") == "curriculum_train_validation_only":
                candidates.append(("expanded", manifest.parent))
    if not candidates:
        raise FileNotFoundError("Add Input the curriculum bundle ZIP or its expanded layout.")
    if len(candidates) != 1:
        raise ValueError("Multiple curriculum inputs found; set BUNDLE_PATH explicitly.")
    kind, path = candidates[0]
    checked = path if kind == "zip" else path / "bundle_manifest.json"
    expected = BUNDLE_SHA256 if kind == "zip" else MANIFEST_SHA256
    if not checked.is_file() or file_sha256(checked) != expected:
        raise ValueError("Curriculum input SHA256 mismatch: " + str(checked))
    return kind, path


def resolve_sft_checkpoint(input_root, expected_sha256, explicit_path=""):
    if explicit_path:
        path = Path(explicit_path)
        if not path.is_file():
            raise FileNotFoundError("Explicit SFT500 path does not exist: " + str(path))
        candidates = [path]
    else:
        candidates = sorted(
            (p for p in Path(input_root).rglob("*.pt") if p.is_file()),
            key=lambda p: (p.name not in ("sft500-final.pt", "final.pt"), str(p)),
        )
    if not candidates:
        raise FileNotFoundError(
            "No .pt weights found. Add Input SFT500 final.pt. "
            "Curriculum bundle and summary ZIP do not contain model weights."
        )
    for path in candidates:
        print("Checking SFT500:", path, "bytes:", path.stat().st_size, flush=True)
        actual = file_sha256(path)
        if actual == expected_sha256:
            print("Verified SFT500:", path, flush=True)
            return path
        print("Checkpoint SHA256:", actual, flush=True)
    raise ValueError("No checkpoint matches SFT500. Expected SHA256: " + expected_sha256)


def prepare_inputs(input_root, scratch_root, explicit_path=""):
    kind, path = find_bundle(input_root, explicit_path)
    if kind == "zip":
        verified = Path(scratch_root) / "verified-bundle"
        run_logged(
            [
                sys.executable,
                "-u",
                "-m",
                "llm_scratch.curriculum.bundle",
                "extract",
                "--archive",
                path,
                "--output",
                verified,
                "--sha256",
                BUNDLE_SHA256,
            ]
        )
    else:
        verified = path
        run_logged(
            [
                sys.executable,
                "-u",
                "-c",
                "import sys; from pathlib import Path; "
                "from llm_scratch.curriculum.bundle import verify; "
                "print('Verified curriculum:', verify(Path(sys.argv[1])), flush=True)",
                path,
            ]
        )
    return verified


INPUTS = prepare_inputs(
    INPUT_ROOT, Path(tempfile.mkdtemp(prefix="grast-curriculum-data-")), BUNDLE_PATH
)
INIT_WEIGHTS = None
if not RESUME:
    INIT_WEIGHTS = resolve_sft_checkpoint(INPUT_ROOT, SFT_SHA256, SFT_CHECKPOINT)
print("Verified curriculum:", INPUTS, flush=True)

In [ ]:
def select_precision(capability_major, bf16_supported, saved_precision=""):
    if saved_precision:
        if saved_precision not in ("float32", "float16", "bfloat16"):
            raise ValueError("Unknown resume precision")
        if saved_precision == "bfloat16" and not (capability_major >= 8 and bf16_supported):
            raise RuntimeError("Checkpoint requires native BF16 GPU for exact resume.")
        return saved_precision
    return "bfloat16" if capability_major >= 8 and bf16_supported else "float16"


def resume_settings(path):
    import torch

    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            "Attach previous Notebook Output and set RESUME to latest.pt: " + str(path)
        )
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    state = payload.get("state", {})
    if (
        payload.get("format_version") != 1
        or state.get("data_state", {}).get("curriculum_schema_version") != 1
    ):
        raise ValueError(
            "RESUME needs full curriculum latest.pt; selected_model.pt/old SFT cannot resume."
        )
    print("Checkpoint runtime:", state["runtime"], flush=True)
    return state["training_config"]["precision"], state["step"]


def gpu_preflight(output_root, saved_precision=""):
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Choose a GPU accelerator; no CPU fallback.")
    capability = torch.cuda.get_device_capability(0)
    precision = select_precision(capability[0], torch.cuda.is_bf16_supported(), saved_precision)
    try:
        matrix = torch.randn((64, 64), device="cuda:0", requires_grad=True)
        with torch.autocast(
            "cuda", dtype=getattr(torch, precision), enabled=precision != "float32"
        ):
            loss = (matrix @ matrix).float().square().mean()
        loss.backward()
        torch.cuda.synchronize(0)
        if not torch.isfinite(loss) or not torch.isfinite(matrix.grad).all():
            raise FloatingPointError("Nonfinite GPU probe")
    except Exception as error:
        raise RuntimeError(
            "GPU forward/backward probe failed; inspect runtime image/driver above."
        ) from error
    report = {
        "device": "cuda:0",
        "gpu": torch.cuda.get_device_name(0),
        "visible_gpus": torch.cuda.device_count(),
        "capability": list(capability),
        "torch": str(torch.__version__),
        "cuda": torch.version.cuda,
        "precision": precision,
        "source_commit": SOURCE_COMMIT,
        "forward_backward_probe": "passed",
        "uses_distributed_training": False,
    }
    Path(output_root).mkdir(parents=True, exist_ok=True)
    (Path(output_root) / "kaggle_environment.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    print(json.dumps(report), flush=True)
    return precision


RESUME_STEP = 0
SAVED_PRECISION = ""
if RESUME:
    SAVED_PRECISION, RESUME_STEP = resume_settings(RESUME)
PRECISION = gpu_preflight(WORK_ROOT, SAVED_PRECISION)

## Train / tiếp tục

`SMOKE_ONLY=True`: lượt mới tới step2; resume chạy thêm tối đa2 successful updates. `SMOKE_ONLY=False`: chạy tới2800 theo recipe cố định. Full validation có log mỗi50 batches, generated validation mỗi200 updates/cuối stage. Stage chuyển theo successful steps, không tự thay budget theo điểm.

Resume giữ precision của checkpoint; loader còn kiểm runtime/GPU count/Torch/source/recipe/data/RNG. Nếu khác môi trường thì dừng với lỗi, không tự bỏ optimizer để ép resume. Save & Run All phải gắn đầy đủ Input trước khi chạy. Hai GPU T4 vẫn chỉ dùng một GPU.


In [ ]:
def training_command(
    repo,
    inputs,
    output_root,
    precision,
    smoke_only,
    *,
    resume="",
    init_checkpoint="",
    resume_step=0,
    device="cuda",
    recipe_path=None,
):
    if precision not in ("float32", "float16", "bfloat16"):
        raise ValueError("Invalid precision")
    if type(resume_step) is not int or not 0 <= resume_step <= TOTAL_UPDATES:
        raise ValueError("Invalid resume step")
    recipe = (
        Path(recipe_path)
        if recipe_path is not None
        else Path(repo) / "configs/training/wp07_curriculum.toml"
    )
    command = [
        sys.executable,
        "-u",
        "-m",
        "llm_scratch.curriculum.runner",
        "--config",
        recipe,
        "--prepared-root",
        Path(inputs) / "curriculum_data",
        "--tokenizer",
        Path(inputs) / "tokenizer",
        "--device",
        device,
        "--precision",
        precision,
        "--output-root",
        Path(output_root),
    ]
    selected = Path(resume) if resume else Path(init_checkpoint)
    if not selected.is_file():
        raise FileNotFoundError(
            "Attach weights/checkpoint and set the exact path: " + str(selected)
        )
    command += ["--resume" if resume else "--init-checkpoint", selected]
    if smoke_only:
        command += ["--stop-after", str(min(resume_step + 2, TOTAL_UPDATES) if resume else 2)]
    return command


LAST_RUN_DIRECTORY = None
command = training_command(
    REPO,
    INPUTS,
    OUTPUT_ROOT,
    PRECISION,
    SMOKE_ONLY,
    resume=RESUME,
    init_checkpoint=INIT_WEIGHTS,
    resume_step=RESUME_STEP,
)
run_logged(command, cwd=REPO)

In [ ]:
import zipfile

from IPython.display import Image, display

if LAST_RUN_DIRECTORY is None:
    raise RuntimeError(
        "No curriculum run started; inspect console.log instead of selecting an old run."
    )
RUN = Path(LAST_RUN_DIRECTORY)
REPORT_PATH = RUN / "artifacts/curriculum_report.json"
if REPORT_PATH.is_file():
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    print(json.dumps(report, indent=2), flush=True)
    if not report["improved_over_initial"]:
        print(
            "No recorded validation improvement; execution success is not a quality pass.",
            flush=True,
        )
for name in ("generated_accuracy.png", "loss_curves.png"):
    path = RUN / "artifacts" / name
    if path.is_file():
        display(Image(filename=str(path)))

SUMMARY = WORK_ROOT / (RUN.name + "-summary.zip")
with zipfile.ZipFile(SUMMARY, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and path.suffix not in {".pt", ".pth", ".ckpt", ".u32"}:
            archive.write(path, path.relative_to(RUN))
    for path in (CONSOLE_LOG, WORK_ROOT / "kaggle_environment.json"):
        if path.is_file():
            archive.write(path, "kaggle/" + path.name)
selected = RUN / "checkpoints/selected_model.pt"
print("Selected weights for next evaluation:", selected, flush=True)
print("Resume checkpoint:", RUN / "checkpoints/latest.pt", flush=True)
if selected.is_file():
    print("Selected-model SHA256:", file_sha256(selected), flush=True)
print("Summary ZIP (no weights):", SUMMARY, flush=True)
print("Keep selected_model.pt AND summary ZIP from Notebook Output for the next phase.", flush=True)

## Output cần giữ

Mọi kết quả nằm trong `/kaggle/working/grast-wp07-curriculum/`: console log, environment report, runs/checkpoints/curves và summary ZIP. Source/data tạm nằm ngoài working. Save Version lưu Output của phiên chạy; nếu runtime bị kill trước khi Output được lưu thì không mặc định coi checkpoint là backup đã bền vững.

Đánh giá tiếp: giữ **selected_model.pt + summary ZIP**. Tiếp tục training: Add Input output version trước, giữ bundle, đặt `RESUME` tới **latest.pt** (không cần SFT500 gốc cho resume). Không mount Drive, không dùng nút upload/download trong phiên nền. Cần xác nhận chạy thực tế Kaggle trước khi gọi đây là nghiệm thu Kaggle; local verification không thay kết quả trên Kaggle GPU.
